# 03 — Satisfaction Driver Analysis

Logistic regression predicting **high revisit intent** from standardized experience ratings.

**Core logic:** `src/driver_analysis.py` — notebook demonstrates target creation and reads model outputs.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from config import EXPERIENCE_COLS
from driver_analysis import train_high_revisit_model

TABLES = ROOT / "outputs" / "tables"
PROCESSED = ROOT / "data" / "processed"

## Target creation

Binary outcome: `high_revisit_intent = 1` when `revisit_intent >= 4`, else `0`. Defined in `train_high_revisit_model()`.

In [ ]:
surveys = pd.read_parquet(PROCESSED / "guest_surveys_classified.parquet")
surveys["high_revisit_intent"] = (surveys["revisit_intent"] >= 4).astype(int)

print(f"Surveys: {len(surveys):,}")
print(f"High revisit intent rate: {surveys['high_revisit_intent'].mean():.1%}")
surveys["revisit_intent"].value_counts().sort_index()

In [ ]:
pd.crosstab(surveys["revisit_intent"], surveys["high_revisit_intent"], margins=True)

## Model results

Precomputed outputs from `make build`. Optionally re-fit inline via `train_high_revisit_model()` (same function the pipeline uses).

In [ ]:
drivers = pd.read_csv(TABLES / "driver_importance.csv")
metrics = pd.read_csv(TABLES / "model_metrics.csv")
metrics

In [ ]:
# Optional: re-run model from src (should match precomputed tables)
drivers_live, metrics_live = train_high_revisit_model(surveys)
print("Live vs saved ROC-AUC:", metrics_live.iloc[0]["roc_auc"], "vs", metrics.iloc[0]["roc_auc"])

## Driver interpretation

Coefficients and odds ratios are on **standardized** experience ratings. Odds ratio > 1 means higher rating associates with higher odds of declaring intent to return.

In [ ]:
drivers[["rank", "driver", "model_coefficient", "odds_ratio", "plain_english_interpretation"]]

In [ ]:
plot_df = drivers.sort_values("odds_ratio").copy()
plot_df["driver_label"] = plot_df["driver"].str.replace("_rating", "").str.replace("_", " ").str.title()

fig = px.bar(
    plot_df,
    x="odds_ratio",
    y="driver_label",
    orientation="h",
    title="Revisit Intent Drivers (Odds Ratios, std-scaled features)",
    color="odds_ratio",
    color_continuous_scale=["#F5F5F5", "#6F4E37"],
)
fig.add_vline(x=1.0, line_dash="dash")
fig.show()

## Takeaway

- **Drink quality rating** is the strongest driver (highest odds ratio).
- Model metrics: accuracy ~84%, ROC-AUC ~92% on held-out test split.
- **Association, not causality** — improving a rating may correlate with revisit intent but requires a pilot to prove causal impact. See `reports/methodology.md`.